In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../data/processed/creditwise_cleaned.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (12000, 44)


In [3]:
sim_df = df[
    [
        "applicant_id",
        "credit_score",
        "debt_to_income_ratio",
        "credit_utilization_ratio",
        "late_payments_24m",
        "annual_income",
        "years_employed",
        "requested_credit_limit",
        "total_credit_exposure",
        "disposable_income",
        "product_type"
    ]
].copy()

print(sim_df.shape)

(12000, 11)


In [4]:
sim_df["dti_flag"] = np.where(
    sim_df["debt_to_income_ratio"] > 0.50,
    1,
    0
)

sim_df["utilization_flag"] = np.where(
    sim_df["credit_utilization_ratio"] > 0.75,
    1,
    0
)

sim_df["late_payment_flag"] = np.where(
    sim_df["late_payments_24m"] >= 3,
    1,
    0
)

sim_df["credit_score_flag"] = np.where(
    sim_df["credit_score"] < 580,
    1,
    0
)

In [5]:
risk_flags = [
    "dti_flag",
    "utilization_flag",
    "late_payment_flag",
    "credit_score_flag"
]

sim_df["risk_indicator_count"] = sim_df[
    risk_flags
].sum(axis=1)

sim_df["risk_segment"] = np.select(
    [
        sim_df["risk_indicator_count"] == 0,
        sim_df["risk_indicator_count"] == 1,
        sim_df["risk_indicator_count"] >= 2
    ],
    [
        "Low Indicator",
        "Moderate Indicator",
        "High Indicator"
    ],
    default="Unknown"
)

sim_df[
    [
        "applicant_id",
        "risk_indicator_count",
        "risk_segment"
    ]
].head(10)

,applicant_id,risk_indicator_count,risk_segment
0,100000,0,Low Indicator
1,100001,0,Low Indicator
2,100002,1,Moderate Indicator
3,100003,0,Low Indicator
4,100004,0,Low Indicator
5,100005,0,Low Indicator
6,100006,0,Low Indicator
7,100007,0,Low Indicator
8,100008,0,Low Indicator
9,100009,0,Low Indicator


In [6]:
sim_df["requested_exposure"] = (
    sim_df["requested_credit_limit"]
)

In [7]:
risk_summary = (
    sim_df
    .groupby("risk_segment")
    .agg(
        applications=("applicant_id", "count"),
        total_requested_exposure=("requested_exposure", "sum"),
        average_requested_exposure=("requested_exposure", "mean"),
        average_credit_score=("credit_score", "mean"),
        average_dti=("debt_to_income_ratio", "mean")
    )
    .reset_index()
)

risk_summary

,risk_segment,applications,total_requested_exposure,average_requested_exposure,average_credit_score,average_dti
0,High Indicator,43,2.440643e+06,56759.139070,542.953488,0.373349
1,Low Indicator,10895,6.655578e+08,61088.375104,663.377513,0.223481
2,Moderate Indicator,1062,6.272028e+07,59058.645998,563.051789,0.272774


In [8]:
product_exposure = (
    sim_df
    .groupby("product_type")
    .agg(
        applications=("applicant_id", "count"),
        total_requested_exposure=("requested_exposure", "sum"),
        average_requested_exposure=("requested_exposure", "mean")
    )
    .reset_index()
)

product_exposure

,product_type,applications,total_requested_exposure,average_requested_exposure
0,Credit Card,7759,4.750693e+08,61228.156482
1,Credit Line,1213,7.410998e+07,61096.442193
2,Personal Loan,3028,1.815395e+08,59953.606760


In [9]:
sim_df["broad_policy"] = (
    sim_df["risk_indicator_count"] <= 2
)

sim_df["balanced_policy"] = (
    sim_df["risk_indicator_count"] <= 1
)

sim_df["strict_policy"] = (
    sim_df["risk_indicator_count"] == 0
)

In [10]:
scenario_summary = pd.DataFrame({
    "Scenario": [
        "Broad Screening",
        "Balanced Screening",
        "Strict Screening"
    ],
    "Eligible Applications": [
        sim_df["broad_policy"].sum(),
        sim_df["balanced_policy"].sum(),
        sim_df["strict_policy"].sum()
    ],
    "Requested Exposure": [
        sim_df.loc[
            sim_df["broad_policy"],
            "requested_exposure"
        ].sum(),

        sim_df.loc[
            sim_df["balanced_policy"],
            "requested_exposure"
        ].sum(),

        sim_df.loc[
            sim_df["strict_policy"],
            "requested_exposure"
        ].sum()
    ]
})

scenario_summary

,Scenario,Eligible Applications,Requested Exposure
0,Broad Screening,11997,7.304343e+08
1,Balanced Screening,11957,7.282781e+08
2,Strict Screening,10895,6.655578e+08


In [11]:
historical_context = (
    sim_df.assign(
        historical_approved=df["approved"].values
    )
    .groupby("risk_segment")
    .agg(
        applications=("applicant_id", "count"),
        historical_approval_rate=(
            "historical_approved",
            "mean"
        )
    )
    .reset_index()
)

historical_context[
    "historical_approval_rate"
] = (
    historical_context[
        "historical_approval_rate"
    ] * 100
)

historical_context

,risk_segment,applications,historical_approval_rate
0,High Indicator,43,0.000000
1,Low Indicator,10895,41.358421
2,Moderate Indicator,1062,24.105461


In [12]:
total_exposure = sim_df[
    "requested_exposure"
].sum()

risk_summary["exposure_share_percent"] = (
    risk_summary["total_requested_exposure"]
    / total_exposure
    * 100
)

risk_summary

,risk_segment,applications,total_requested_exposure,average_requested_exposure,average_credit_score,average_dti,exposure_share_percent
0,High Indicator,43,2.440643e+06,56759.139070,542.953488,0.373349,0.334006
1,Low Indicator,10895,6.655578e+08,61088.375104,663.377513,0.223481,91.082626
2,Moderate Indicator,1062,6.272028e+07,59058.645998,563.051789,0.272774,8.583368


In [13]:
risk_summary.to_csv(
    "../data/outputs/risk_segment_simulation.csv",
    index=False
)

print("Risk segment simulation saved.")

Risk segment simulation saved.


In [14]:
scenario_summary.to_csv(
    "../data/outputs/risk_policy_scenarios.csv",
    index=False
)

print("Policy scenario results saved.")

Policy scenario results saved.


In [15]:
sim_df.to_csv(
    "../data/outputs/applicant_risk_simulation.csv",
    index=False
)

print("Applicant-level risk simulation saved.")

Applicant-level risk simulation saved.
